In [ ]:
# =============================================================================
# Short Insights — 공매도 데이터에서 인사이트 추출
# =============================================================================
#
# short_data/ 의 종목별 CSV를 읽어서 8가지 인사이트 테이블을 생성:
#
#   1. 잔고비중 Top 20      — 공매도 잔고가 발행주식수 대비 가장 높은 종목
#   2. 공매도비중 Top 20     — 당일 공매도 거래가 전체 거래 대비 가장 높은 종목
#   3. 공매도금액 Top 20     — 공매도 금액 절대값이 가장 큰 종목
#   4. 비중 급증 Top 10      — 전일 대비 공매도비중이 가장 많이 오른 종목
#   5. 비중 급감 Top 10      — 전일 대비 공매도비중이 가장 많이 내린 종목
#   6. 252일 최고 근접       — 공매도비중이 1년 내 역대 최고에 근접한 종목
#   7. 252일 최저 근접       — 공매도비중이 1년 내 역대 최저에 근접한 종목
#   8. Squeeze Watch Top 20 — 5가지 시그널 종합 스코어 (숏스퀴즈 후보)
#
# 원본 short_insights.py 대비 수정 사항:
#   FIX #4: 컬럼명 한→영 rename을 로드 직후 즉시 적용 (원본은 나중에 해서 num_cols 체크 실패)
#   FIX #5: 거래량 0 종목 필터 (거래정지, 상폐 등 → division by zero 방지)
#   FIX #6: MA 이격률 NaN 전파 방지 (px_last/ma_20d NaN이면 0으로 처리)
#   FIX #7: Bloomberg 없을 때 free_float NaN → 100%로 fallback (score NaN 방지)
#   FIX #8: Movers에서 실제 비교 날짜 표시 (공휴일 gap 시 "day-over-day"가 아닐 수 있음)
# =============================================================================

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd

REPO_DIR = Path("short_data")

# pykrx CSV는 한국어 컬럼명 → 영어로 통일
COL_RENAME = {
    "공매도": "short_vol",        # 공매도 거래량 (주)
    "매수": "buy_vol",            # 총 매수 거래량 (주)
    "공매도비중": "short_vol_pct", # 공매도 / 매수 비율 (%)
    "공매도잔고": "short_bal",     # 미결제 공매도 수량 (주)
    "상장주식수": "listed_shares", # 발행주식수
    "공매도금액": "short_notional",# 공매도잔고 × 가격 (KRW)
    "시가총액": "market_cap",      # 시가총액 (KRW)
    "잔고비중": "bal_pct",         # 공매도잔고 / 상장주식수 (%)
    "시장": "market",              # KOSPI / KOSDAQ
}

# 숫자로 변환해야 하는 컬럼들 (CSV에서 읽으면 문자열일 수 있음)
NUM_COLS = ["short_vol", "buy_vol", "short_vol_pct", "short_bal", "listed_shares",
            "short_notional", "market_cap", "bal_pct", "px_last", "free_float", "index_px"]

In [ ]:
# =============================================================================
# 데이터 로드 — ~2500개 종목별 CSV를 하나의 DataFrame으로 합침
# =============================================================================
#
# 결과: "tall" DataFrame (한 행 = 한 종목의 하루)
#   약 2500 종목 × 252일 = ~63만 행
#   로드 시간: ~1.5초

def load_repo(repo_dir: Path = REPO_DIR, n_days: int | None = 252) -> pd.DataFrame:
    """종목별 CSV를 전부 읽어서 하나의 tall DataFrame으로 합침.

    n_days: 종목당 최근 N일만 로드 (퍼센타일 계산에 252일 필요).
           None이면 전체 로드.
    """
    csvs = sorted(repo_dir.glob("*.csv"))
    if not csvs:
        print(f"데이터 없음: {repo_dir}")
        return pd.DataFrame()

    def _read(p):
        try:
            df = pd.read_csv(p, dtype={"date": str})
            if df.empty:
                return None
            df["ticker"] = p.stem  # 파일명이 곧 종목코드 (예: 005930)
            return df
        except Exception:
            return None  # 깨진 파일은 skip

    print(f"Loading {len(csvs)} stock files...")
    # ThreadPoolExecutor로 병렬 읽기 (I/O bound → 멀티스레드가 효과적)
    with ThreadPoolExecutor(max_workers=8) as pool:
        frames = [f for f in pool.map(_read, csvs) if f is not None]

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)

    # FIX #4: rename을 여기서 즉시 적용
    # 원본은 generate_insights()에서만 rename해서, load_repo()의 num_cols 체크가
    # 한국어 컬럼에 적용되어 실질적으로 무시되었음
    df = df.rename(columns=COL_RENAME)
    df = df.sort_values(["ticker", "date"])

    # 종목당 최근 n_days만 유지 (메모리 절약 + 불필요한 오래된 데이터 제외)
    if n_days:
        df = df.groupby("ticker", sort=False).tail(n_days)

    # 문자열 → 숫자 변환 (CSV에서 읽으면 가끔 문자열로 들어옴)
    for col in NUM_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    print(f"Loaded: {len(df):,} rows, {df['ticker'].nunique():,} stocks, "
          f"{df['date'].nunique()} dates ({df['date'].min()} → {df['date'].max()})")
    return df.reset_index(drop=True)


# ── 로드 실행 ──
df = load_repo()
DATES = sorted(df["date"].unique()) if not df.empty else []
LATEST = DATES[-1] if DATES else ""  # 가장 최근 거래일
print(f"Latest: {LATEST}")

In [ ]:
# =============================================================================
# 1. 잔고비중 Top 20 — 공매도 잔고가 발행주식수 대비 가장 높은 종목
# =============================================================================
#
# 잔고비중 = 공매도잔고 / 상장주식수 (%)
# 이 값이 높을수록 "숏이 많이 쌓여 있다" = 스퀴즈 잠재력 ↑
#
# 주의: short_bal은 ~3영업일 지연 공시됨.
# → bal_pct가 NaN이 아닌 가장 최근 날짜를 사용 (LATEST와 다를 수 있음)
# =============================================================================

# bal_pct가 있는 가장 최근 날짜 찾기 (short_bal은 지연 공시라 LATEST보다 이전일 수 있음)
bal_dates = sorted(df.dropna(subset=["bal_pct"])["date"].unique())
BAL_LATEST = bal_dates[-1] if bal_dates else LATEST

day_bal = df[df["date"] == BAL_LATEST].dropna(subset=["bal_pct"]).copy()

# FIX #5: bal_pct == 0인 종목 제외 (공매도 잔고 자체가 없는 종목은 의미 없음)
day_bal = day_bal[day_bal["bal_pct"] > 0]

bal_top = day_bal.nlargest(20, "bal_pct")[
    ["ticker", "market", "bal_pct", "short_bal", "listed_shares", "short_notional", "market_cap"]
].copy()

# 억원 단위로 변환 (원 단위는 읽기 어려움)
bal_top["short_notional_억"] = (bal_top["short_notional"] / 1e8).round(1)
bal_top["market_cap_억"] = (bal_top["market_cap"] / 1e8).round(0)

print(f"잔고비중 Top 20 — {BAL_LATEST} (short_bal 기준일)")
display(bal_top[["ticker", "market", "bal_pct", "short_bal", "short_notional_억", "market_cap_억"]]
    .reset_index(drop=True)
    .style.format({"bal_pct": "{:.2f}%", "short_bal": "{:,.0f}",
                   "short_notional_억": "{:,.1f}", "market_cap_억": "{:,.0f}"}))

In [ ]:
# =============================================================================
# 2. 공매도비중 Top 20 — 당일 공매도 거래량 / 전체 거래량
# =============================================================================
#
# 공매도비중 = 공매도 거래량 / 매수 거래량 (%)
# 높으면 "오늘 이 종목 거래의 상당 부분이 공매도였다"
# → 누군가가 적극적으로 숏을 치고 있다는 신호
# =============================================================================

day_vol = df[df["date"] == LATEST].dropna(subset=["short_vol_pct"]).copy()

# FIX #5: 거래량 0인 종목 제외
# → 거래정지, 상폐, 또는 전혀 거래가 없는 종목은 공매도비중이 의미 없음
# → buy_vol == 0이면 short_vol_pct도 0/NaN이지만, 명시적으로 필터
day_vol = day_vol[day_vol["buy_vol"] > 0]

vol_top = day_vol.nlargest(20, "short_vol_pct")[
    ["ticker", "market", "short_vol_pct", "short_vol", "buy_vol", "px_last"]
].reset_index(drop=True)

print(f"공매도비중 Top 20 — {LATEST}")
display(vol_top.style.format({
    "short_vol_pct": "{:.2f}%", "short_vol": "{:,.0f}",
    "buy_vol": "{:,.0f}", "px_last": "{:,.0f}"
}))

In [ ]:
# =============================================================================
# 3. 공매도금액 Top 20 — 공매도 잔고의 절대 금액 (억원)
# =============================================================================
#
# 비중이 아닌 절대 금액 기준 → 대형주 위주로 나옴
# "실제로 얼마나 많은 돈이 숏에 걸려 있는가"
# =============================================================================

day_notional = df[df["date"] == LATEST].dropna(subset=["short_notional"]).copy()

# 잔고가 0인 종목 제외
day_notional = day_notional[day_notional["short_notional"] > 0]

day_notional["short_notional_억"] = (day_notional["short_notional"] / 1e8).round(1)

# FIX #5: buy_vol == 0이면 total_traded_억을 NaN으로 (0으로 나누기 방지)
day_notional["total_traded_억"] = np.where(
    day_notional["buy_vol"] > 0,
    (day_notional["buy_vol"] * day_notional["px_last"] / 1e8).round(1),
    np.nan
)

notional_top = day_notional.nlargest(20, "short_notional")[
    ["ticker", "market", "short_notional_억", "total_traded_억", "short_vol_pct"]
].reset_index(drop=True)

print(f"공매도금액 Top 20 — {LATEST}")
display(notional_top.style.format({
    "short_notional_억": "{:,.1f}", "total_traded_억": "{:,.1f}",
    "short_vol_pct": "{:.2f}%"
}))

In [ ]:
# =============================================================================
# 4+5. 비중 급증/급감 Top 10 — 전일 대비 공매도비중 변화
# =============================================================================
#
# 공매도비중이 하루 사이에 크게 변한 종목을 찾음.
# 급증 = 누군가 갑자기 공매도를 많이 침
# 급감 = 숏커버링(공매도 청산) 또는 거래 자체가 줄어듦
#
# FIX #8: DATES[-1]과 DATES[-2]는 실제 거래일이므로 공휴일이 끼면
#         "day-over-day"가 아닌 2~3일 gap일 수 있음 → 날짜를 명시적으로 표시
# =============================================================================

if len(DATES) >= 2:
    today_str, prev_str = DATES[-1], DATES[-2]

    # 당일/전일의 공매도비중 데이터
    today_sr = df[df["date"] == today_str][["ticker", "market", "short_vol_pct"]].copy()
    prev_sr = df[df["date"] == prev_str][["ticker", "short_vol_pct"]].copy()

    # FIX #5: NaN인 종목 제외 (거래량 0 → 비중도 의미 없음)
    today_sr = today_sr.dropna(subset=["short_vol_pct"])
    prev_sr = prev_sr.dropna(subset=["short_vol_pct"])

    # 양일 모두 데이터가 있는 종목만 비교 (inner join)
    movers = today_sr.merge(prev_sr, on="ticker", suffixes=("_today", "_prev"))
    movers["change_pp"] = (movers["short_vol_pct_today"] - movers["short_vol_pct_prev"]).round(3)

    # 급증 Top 10
    print(f"비중 급증 Top 10 — {prev_str} → {today_str}")
    display(movers.nlargest(10, "change_pp")[
        ["ticker", "market", "short_vol_pct_prev", "short_vol_pct_today", "change_pp"]
    ].reset_index(drop=True).style.format({
        "short_vol_pct_prev": "{:.2f}%", "short_vol_pct_today": "{:.2f}%",
        "change_pp": "{:+.2f}pp"
    }))

    # 급감 Top 10
    print(f"\n비중 급감 Top 10 — {prev_str} → {today_str}")
    display(movers.nsmallest(10, "change_pp")[
        ["ticker", "market", "short_vol_pct_prev", "short_vol_pct_today", "change_pp"]
    ].reset_index(drop=True).style.format({
        "short_vol_pct_prev": "{:.2f}%", "short_vol_pct_today": "{:.2f}%",
        "change_pp": "{:+.2f}pp"
    }))
else:
    print("날짜가 2일 미만이라 비교 불가")

In [ ]:
# =============================================================================
# 6+7. 252일 퍼센타일 (High / Low) — 공매도비중의 1년 내 상대 위치
# =============================================================================
#
# 각 종목의 공매도비중이 최근 252 거래일(≈1년) 내에서 어느 위치에 있는지 계산.
# 100%에 가까우면 = 1년 내 역대 최고 근접 (공매도가 역사적으로 많은 상태)
# 0%에 가까우면 = 1년 내 역대 최저 근접 (공매도가 거의 없는 상태)
#
# rank-based percentile 사용 (min-max 대비 이상치에 강건함)
# =============================================================================

def compute_percentiles(df: pd.DataFrame, col: str = "short_vol_pct", n: int = 10):
    """252일 rank-based 퍼센타일. (near_high, near_low) 반환."""
    recent = df.dropna(subset=[col]).sort_values("date")
    recent = recent.groupby("ticker", sort=False).tail(252)

    def _pctl(group):
        # 최소 20일 데이터 필요 (신규상장/거래재개 직후는 불안정)
        if len(group) < 20:
            return None

        latest_val = group.iloc[-1][col]

        # FIX #5: 거래량 0으로 인해 비중이 거의 항상 0인 종목은 제외
        # (예: 우선주, ETN 등 → 50% 이상이 0이면 의미 없는 데이터)
        if latest_val == 0 and (group[col] == 0).sum() > len(group) * 0.5:
            return None

        # rank-based percentile: 현재 값보다 작거나 같은 관측값의 비율
        pctl = (group[col] <= latest_val).sum() / len(group) * 100

        return pd.Series({
            "market": group.iloc[-1]["market"],
            "today_val": latest_val,
            "252d_low": group[col].min(),
            "252d_high": group[col].max(),
            "percentile": round(pctl, 1),
        })

    stats = recent.groupby("ticker").apply(_pctl, include_groups=False)
    stats = stats.dropna(subset=["percentile"]).reset_index()

    near_high = stats.nlargest(n, "percentile")   # 최고 근접
    near_low = stats.nsmallest(n, "percentile")    # 최저 근접
    return near_high, near_low


near_high, near_low = compute_percentiles(df)

print(f"252일 최고 근접 Top 10 (short_vol_pct 기준)")
display(near_high[["ticker", "market", "today_val", "252d_low", "252d_high", "percentile"]]
    .reset_index(drop=True)
    .style.format({"today_val": "{:.2f}%", "252d_low": "{:.2f}%",
                   "252d_high": "{:.2f}%", "percentile": "{:.0f}th"}))

print(f"\n252일 최저 근접 Top 10 (short_vol_pct 기준)")
display(near_low[["ticker", "market", "today_val", "252d_low", "252d_high", "percentile"]]
    .reset_index(drop=True)
    .style.format({"today_val": "{:.2f}%", "252d_low": "{:.2f}%",
                   "252d_high": "{:.2f}%", "percentile": "{:.0f}th"}))

In [ ]:
# =============================================================================
# 8. Squeeze Watch Top 20 — 숏스퀴즈 후보 종합 스코어
# =============================================================================
#
# 5가지 시그널을 결합해 스퀴즈 가능성이 높은 종목을 스크리닝:
#
#   (1) 잔고 퍼센타일 (25점)
#       → 잔고비중이 252일 역사 대비 상위에 위치 (숏이 많이 쌓임)
#
#   (2) 커버링 강도 (25점)
#       → 최근 10일간 공매도비중이 감소 추세 (숏커버링 시작 신호)
#       → first-3일 평균 vs last-3일 평균의 차이 (pp)
#
#   (3) MA 돌파 (15점, sigmoid)
#       → 주가가 20일 이동평균 위에 있으면 상승 모멘텀
#       → sigmoid 함수로 경계값 근처에서도 부분 점수 부여
#
#   (4) 거래량 급증 (25점)
#       → 당일 거래량 / 20일 평균 거래량 (1.5x 이상이면 관심 급증)
#
#   (5) 유동주식 대비 공매도 (10점)
#       → short_bal / (listed_shares × free_float%) — 실질적 숏 압력
#       → Bloomberg 없으면 free_float = 100% 가정 (보수적 추정)
#
# 모든 시그널이 연속 함수 → 경계값 부근에서도 부분 점수 부여
# =============================================================================

MIN_MARKET_CAP = 1e11  # 1000억원 — micro-cap 제외 (노이즈 방지)


def squeeze_watch(df: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    """Composite squeeze score. Returns top N candidates."""

    # ── (1) 잔고비중 252일 퍼센타일 ──────────────────────────────────
    bal_recent = df.dropna(subset=["bal_pct"]).sort_values("date")
    bal_recent = bal_recent.groupby("ticker", sort=False).tail(252)

    def _bal_pctl(g):
        if len(g) < 20:
            return None  # 데이터 부족한 종목 제외
        last = g.iloc[-1]
        pctl = (g["bal_pct"] <= last["bal_pct"]).sum() / len(g) * 100
        return pd.Series({
            "market": last["market"], "bal_pct": last["bal_pct"],
            "bal_percentile": pctl,
            "short_bal": last.get("short_bal", np.nan),
            "listed_shares": last.get("listed_shares", np.nan),
            "market_cap": last.get("market_cap", np.nan),
        })

    bal_stats = bal_recent.groupby("ticker").apply(_bal_pctl, include_groups=False)
    bal_stats = bal_stats.dropna(subset=["bal_percentile"])

    # 70th 퍼센타일 이상만 후보 (역사적으로 숏이 많이 쌓인 종목만)
    candidates = bal_stats[bal_stats["bal_percentile"] >= 70].copy()
    if candidates.empty:
        print("bal_percentile >= 70인 종목 없음")
        return pd.DataFrame()

    # 시가총액 필터 — FIX #5: NaN/0 처리
    candidates["market_cap"] = pd.to_numeric(candidates["market_cap"], errors="coerce").fillna(0)
    candidates = candidates[candidates["market_cap"] >= MIN_MARKET_CAP]
    if candidates.empty:
        print("시가총액 >= 1000억 필터 후 종목 없음")
        return pd.DataFrame()

    # ── (2) 커버링 시그널 (10일 공매도비중 추세) ─────────────────────
    svol_recent = df.dropna(subset=["short_vol_pct"]).sort_values("date")
    svol_recent = svol_recent.groupby("ticker", sort=False).tail(10)

    def _trend(g):
        if len(g) < 4:
            return pd.Series({"vol_trend_5d": np.nan})  # 데이터 부족 → NaN
        # first-3일 평균 vs last-3일 평균의 차이
        return pd.Series({
            "vol_trend_5d": g.tail(3)["short_vol_pct"].mean() - g.head(3)["short_vol_pct"].mean()
        })

    svol_trend = svol_recent.groupby("ticker").apply(_trend, include_groups=False)
    candidates = candidates.join(svol_trend, how="left")
    # NaN이면 0으로 (데이터 부족 → 커버링 시그널 중립)
    candidates["vol_trend_5d"] = candidates["vol_trend_5d"].fillna(0)

    # ── (3) 주가 MA + (4) 거래량 서지 ───────────────────────────────
    px_recent = df.dropna(subset=["px_last"]).sort_values("date")
    px_recent = px_recent.groupby("ticker", sort=False).tail(25)

    latest_px = px_recent.groupby("ticker").tail(1).set_index("ticker")
    last_20 = px_recent.groupby("ticker", sort=False).tail(20)
    ma20 = last_20.groupby("ticker")["px_last"].mean().rename("ma_20d")

    # FIX #5: 거래량 0인 날짜 제외 (거래정지 → 평균 거래량에 0 포함되면 왜곡)
    vol_recent = df.dropna(subset=["buy_vol"]).sort_values("date")
    vol_recent = vol_recent[vol_recent["buy_vol"] > 0]
    vol_recent = vol_recent.groupby("ticker", sort=False).tail(20)
    vol_avg = vol_recent.groupby("ticker")["buy_vol"].mean().rename("avg_buy_vol")

    latest_vol = df[df["buy_vol"] > 0].dropna(subset=["buy_vol"]).sort_values("date")
    latest_vol = latest_vol.groupby("ticker").tail(1).set_index("ticker")

    candidates["px_last"] = latest_px["px_last"].reindex(candidates.index)
    candidates["ma_20d"] = ma20.reindex(candidates.index)

    # FIX #6: px_last나 ma_20d가 NaN이면 sigmoid에서 NaN 전파됨
    # → NaN인 경우 ma_dev_pct = 0 (중립 처리)
    candidates["ma_dev_pct"] = np.where(
        candidates["px_last"].notna() & candidates["ma_20d"].notna() & (candidates["ma_20d"] > 0),
        ((candidates["px_last"] / candidates["ma_20d"]) - 1) * 100,
        0.0  # NaN → 0 = sigmoid 중립점 (7.5/15)
    )

    latest_buy = latest_vol["buy_vol"].reindex(candidates.index).fillna(0)
    avg_buy = vol_avg.reindex(candidates.index).fillna(0)
    # vol_ratio = 당일 거래량 / 20일 평균 거래량. clip(lower=1)로 0 나누기 방지.
    candidates["vol_ratio"] = (latest_buy / avg_buy.clip(lower=1)).replace(
        [np.inf, -np.inf], 0).fillna(0)

    # ── DTC (Days to Cover) ──────────────────────────────────────────
    # 공매도 잔고(주) / 20일 평균 거래량 = 모든 숏을 청산하는 데 필요한 거래일 수
    candidates["short_bal"] = pd.to_numeric(candidates["short_bal"], errors="coerce").fillna(0)
    candidates["dtc"] = (candidates["short_bal"] / avg_buy.reindex(candidates.index).clip(lower=1)
                         ).replace([np.inf, -np.inf], 0).fillna(0).round(1)

    # ── (5) 유동주식 대비 공매도 비중 ────────────────────────────────
    candidates["listed_shares"] = pd.to_numeric(candidates["listed_shares"], errors="coerce").fillna(0)
    ff_pct = latest_px["free_float"].reindex(candidates.index)

    # FIX #7: Bloomberg 없으면 free_float이 전부 NaN
    # → 원본에서는 NaN × listed_shares = NaN → score 전체 NaN → 결과 0건
    # → 해결: NaN이면 100% (전체 유통) 가정. 보수적 추정이지만 score가 동작함.
    ff_pct_safe = ff_pct.fillna(100.0)
    free_float_shares = (candidates["listed_shares"] * ff_pct_safe / 100).clip(lower=1)
    candidates["ff_short_pct"] = (
        (candidates["short_bal"] / free_float_shares) * 100
    ).replace([np.inf, -np.inf], 0).fillna(0).round(2)

    # ── 점수 계산 (총 100점) ─────────────────────────────────────────
    s_bal = candidates["bal_percentile"] * 0.25                     # 0–25
    s_cov = (-candidates["vol_trend_5d"]).clip(0, 3) * (25 / 3)    # 0–25 (감소할수록 높음)
    s_ma  = 15 / (1 + np.exp(-candidates["ma_dev_pct"] * 1.5))     # 0–15 (sigmoid)
    s_vol = (candidates["vol_ratio"] - 1.0).clip(0, 4) * (25 / 4)  # 0–25 (1x 이하 = 0점)
    s_ff  = candidates["ff_short_pct"].clip(0, 10)                  # 0–10

    candidates["score"] = (s_bal + s_cov + s_ma + s_vol + s_ff).round(1)

    # 상위 N개 반환
    result = candidates.nlargest(n, "score").reset_index()
    keep = ["ticker", "market", "bal_pct", "bal_percentile", "vol_trend_5d",
            "px_last", "ma_20d", "ma_dev_pct", "vol_ratio", "dtc", "ff_short_pct", "score"]
    return result[[c for c in keep if c in result.columns]]


# ── 실행 ──
squeeze = squeeze_watch(df)
print(f"Squeeze Watch Top 20 ({len(squeeze)} results)")
if not squeeze.empty:
    display(squeeze.style.format({
        "bal_pct": "{:.2f}%", "bal_percentile": "{:.0f}", "vol_trend_5d": "{:+.2f}pp",
        "px_last": "{:,.0f}", "ma_20d": "{:,.0f}", "ma_dev_pct": "{:+.1f}%",
        "vol_ratio": "{:.1f}x", "dtc": "{:.1f}d", "ff_short_pct": "{:.2f}%", "score": "{:.0f}",
    }).bar(subset=["score"], color="#22c55e44"))

In [ ]:
# =============================================================================
# Excel 저장 — 모든 인사이트를 시트별로 저장
# =============================================================================

insights = {
    "Balance_Top20": bal_top.reset_index(drop=True) if "bal_top" in dir() else pd.DataFrame(),
    "Volume_Top20": vol_top if "vol_top" in dir() else pd.DataFrame(),
    "Notional_Top20": notional_top if "notional_top" in dir() else pd.DataFrame(),
    "Pctl_High": near_high.reset_index(drop=True) if "near_high" in dir() else pd.DataFrame(),
    "Pctl_Low": near_low.reset_index(drop=True) if "near_low" in dir() else pd.DataFrame(),
    "Squeeze_Top20": squeeze if "squeeze" in dir() else pd.DataFrame(),
}

# Movers는 날짜가 2일 이상일 때만 존재
if len(DATES) >= 2 and "movers" in dir():
    insights["Rising_Top10"] = movers.nlargest(10, "change_pp").reset_index(drop=True)
    insights["Falling_Top10"] = movers.nsmallest(10, "change_pp").reset_index(drop=True)

xlsx_path = Path(f"short_insights_{LATEST}.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    for name, table in insights.items():
        if not table.empty:
            table.to_excel(writer, sheet_name=name, index=False)

print(f"Saved: {xlsx_path}")
for name, table in insights.items():
    print(f"  {name:<20s}  {len(table)} rows")